<a href="https://colab.research.google.com/github/jenita-john/EmoLLMs/blob/main/ExperimentalResults.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained('lzw1008/Emot5-large')
model = AutoModelForSeq2SeqLM.from_pretrained('lzw1008/Emot5-large', device_map='auto')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [2]:
def run_task(prompt):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(input_ids, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Task 1: Emotion Detection

Detect prescence of emotions in text:

In [3]:
prompt = """Task: Categorize the text's emotional tone as either 'neutral or no emotion' or identify the presence of one or more of the given emotions (anger, anticipation, disgust, fear, joy, love, optimism, pessimism, sadness, surprise, trust).
Text: Whatever you decide to do make sure it makes you happy.
This text contains emotions:"""

print(run_task(prompt))

joy, optimism.


In [9]:
prompt = """Task: Categorize the text's emotional tone as either 'neutral or no emotion' or identify the presence of one or more of the given emotions (anger, anticipation, disgust, fear, joy, love, optimism, pessimism, sadness, surprise, trust).
Text: There was so much to do today and not enough time.
This text contains emotions:"""

print(run_task(prompt))

anticipation, fear, sadness.


## Task 2: Sentiment Classification

Classify text as positive, negative, or neutral:

In [4]:
prompt = """ Task: Categorize the text into an ordinal class that best characterizes the writer's mental state, considering various degrees of positive and negative sentiment intensity. 3: very positive mental state can be inferred. 2: moderately positive mental state can be inferred. 1: slightly positive mental state can be inferred. 0: neutral or mixed mental state can be inferred. -1: slightly negative mental state can be inferred. -2: moderately negative mental state can be inferred. -3: very negative mental state can be inferred
Text: Beyoncé resentment gets me in my feelings every time. 😩
Intensity Class:"""

print(run_task(prompt))

-2: moderately negative emotional state can be inferred


In [10]:
prompt = """ Task: Categorize the text into an ordinal class that best characterizes the writer's mental state, considering various degrees of positive and negative sentiment intensity. 3: very positive mental state can be inferred. 2: moderately positive mental state can be inferred. 1: slightly positive mental state can be inferred. 0: neutral or mixed mental state can be inferred. -1: slightly negative mental state can be inferred. -2: moderately negative mental state can be inferred. -3: very negative mental state can be inferred
Text: That was one of the best concert's I've ever been to.
Intensity Class:"""

print(run_task(prompt))

3: very positive emotional state can be inferred


##Task 3: Emotion Intensity (Regression)
Return a score from 0 to 1 for a specific emotion:

In [5]:
prompt = """Task: Assign a numerical value between 0 (least E) and 1 (most E) to represent the intensity of emotion E expressed in the text.
Text: I can't stop smiling today, everything is perfect!
Emotion: joy
Intensity Score:"""

print(run_task(prompt))

0.86


In [11]:
prompt = """Task: Assign a numerical value between 0 (least E) and 1 (most E) to represent the intensity of emotion E expressed in the text.
Text: Things have been chaotic recently.
Emotion: joy
Intensity Score:"""

print(run_task(prompt))

0.167


## Task 4: Batch Test Multiple Sentences

In [6]:
sentences = [
    ("I am so angry I could scream.", "anger"),
    ("This is the best news I've heard all year!", "joy"),
    ("I feel completely empty inside.", "sadness"),
]

for text, emotion in sentences:
    prompt = f"""Task: Assign a numerical value between 0 (least E) and 1 (most E) to represent the intensity of emotion E expressed in the text.
Text: {text}
Emotion: {emotion}
Intensity Score:"""
    print(f"Text: {text}")
    print(f"Emotion ({emotion}) intensity: {run_task(prompt)}\n")

Text: I am so angry I could scream.
Emotion (anger) intensity: 0.812

Text: This is the best news I've heard all year!
Emotion (joy) intensity: 0.833

Text: I feel completely empty inside.
Emotion (sadness) intensity: 0.854

